In [12]:
!pip install anthropic chromadb pypdf2


In [13]:
import os

# Point to your PDFs folder in Google Drive
pdf_folder = '/content/drive/MyDrive/hr-policy-copilot/docs'

uploaded = {}
for filename in os.listdir(pdf_folder):
    if filename.endswith('.pdf'):
        with open(os.path.join(pdf_folder, filename), 'rb') as f:
            uploaded[filename] = f.read()

print(f"Loaded {len(uploaded)} PDFs")



Loaded 22 PDFs


In [10]:
!pip install pymupdf

In [14]:
import PyPDF2
import io

# Extract text from all uploaded PDFs
docs = {}
for filename, content in uploaded.items():
    pdf_reader = PyPDF2.PdfReader(io.BytesIO(content))
    text = ""
    for page in pdf_reader.pages:
        text += page.extract_text()
    docs[filename] = text
    print(f"✓ {filename}: {len(text)} characters extracted")

✓ Employment Act 1968.pdf: 192157 characters extracted
✓ Employment of Foreign Manpower Act 1990.pdf: 80448 characters extracted
✓ Retirement and Re-employment Act 1993.pdf: 43014 characters extracted
✓ workright-guide-employment-laws.pdf: 75713 characters extracted
✓ tripartite-guidelines-on-wrongful-dismissal.pdf: 9562 characters extracted
✓ tripartite-guidelines-on-mandatory-retrenchment-notifications.pdf: 3119 characters extracted
✓ tripartite-guidelines-on-re-employment-of-older-employees.pdf: 33317 characters extracted
✓ Tripartite Guidelines.pdf: 37307 characters extracted
✓ tripartite-guidelines-on-flexible-work-arrangement-requests.pdf: 21740 characters extracted
✓ tripartite-advisory-on-managing-excess-manpower-and-responsible-retrenchment.pdf: 25552 characters extracted
✓ tripartite-advisory-on-managing-workplace-harassment.pdf: 27195 characters extracted
✓ tripartite-advisory-on-the-employment-of-term-contract-employees.pdf: 6170 characters extracted
✓ Mental Wellbeing at W

In [15]:
import fitz  # pymupdf

# Re-extract the 0-character files using pymupdf
failed_files = [name for name, text in docs.items() if len(text) == 0]

for filename in failed_files:
    pdf_document = fitz.open(stream=uploaded[filename], filetype="pdf")
    text = ""
    for page in pdf_document:
        text += page.get_text()
    docs[filename] = text
    print(f"✓ {filename}: {len(text)} characters extracted")

✓ Mental Wellbeing at Work.pdf: 0 characters extracted
✓ annual-leave.pdf: 0 characters extracted
✓ SICK LEAVE.pdf: 0 characters extracted
✓ 112.pdf: 0 characters extracted
✓ 123.pdf: 0 characters extracted
✓ 213.pdf: 0 characters extracted
✓ 12312.pdf: 0 characters extracted
✓ 1231.pdf: 0 characters extracted


In [16]:
# Remove empty docs
docs = {name: text for name, text in docs.items() if len(text) > 0}
print(f"Working with {len(docs)} documents")
for name in docs:
    print(f"  - {name}")

Working with 14 documents
  - Employment Act 1968.pdf
  - Employment of Foreign Manpower Act 1990.pdf
  - Retirement and Re-employment Act 1993.pdf
  - workright-guide-employment-laws.pdf
  - tripartite-guidelines-on-wrongful-dismissal.pdf
  - tripartite-guidelines-on-mandatory-retrenchment-notifications.pdf
  - tripartite-guidelines-on-re-employment-of-older-employees.pdf
  - Tripartite Guidelines.pdf
  - tripartite-guidelines-on-flexible-work-arrangement-requests.pdf
  - tripartite-advisory-on-managing-excess-manpower-and-responsible-retrenchment.pdf
  - tripartite-advisory-on-managing-workplace-harassment.pdf
  - tripartite-advisory-on-the-employment-of-term-contract-employees.pdf
  - wr-kets-template-sample-english.pdf
  - tripartite-advisory-on-managing-excess-manpower-and-responsible-retrenchment (1).pdf


In [17]:
def chunk_text(text, chunk_size=300, overlap=100):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks

# Rechunk all docs
all_chunks = []
all_metadata = []

for filename, text in docs.items():
    chunks = chunk_text(text)
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_metadata.append({"source": filename, "chunk": i})

print(f"Total chunks: {len(all_chunks)}")


Total chunks: 300


In [19]:
import chromadb

# Set up ChromaDB
client = chromadb.Client()
collection = client.create_collection("hr_policies")

# Add chunks in batches
batch_size = 50
for i in range(0, len(all_chunks), batch_size):
    batch_chunks = all_chunks[i:i + batch_size]
    batch_metadata = all_metadata[i:i + batch_size]
    batch_ids = [str(j) for j in range(i, i + len(batch_chunks))]

    collection.add(
        documents=batch_chunks,
        metadatas=batch_metadata,
        ids=batch_ids
    )

print(f"✓ {collection.count()} chunks loaded into ChromaDB")

✓ 300 chunks loaded into ChromaDB


In [24]:
import anthropic

claude = anthropic.Anthropic(api_key="YOUR_API_KEY_HERE")

def ask_hr_copilot(question):
    results = collection.query(query_texts=[question], n_results=6)
    context = ""
    sources = []
    for doc, metadata in zip(results["documents"][0], results["metadatas"][0]):
        context += f"\n---\n{doc}"
        sources.append(metadata["source"])

    response = claude.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1000,
        messages=[{
            "role": "user",
            "content": f"You are an HR policy assistant for Singapore employers. Answer based ONLY on the context below. Always cite which document your answer comes from. If the answer is not in the context, say so. Context: {context} Question: {question}"
        }]
    )
    print(f"ANSWER:\n{response.content[0].text}")
    print(f"\nSOURCES: {list(set(sources))}")


In [25]:
test_questions = [
    "What is the notice period required for termination of employment?",
    "How many days of sick leave is an employee entitled to?",
    "What retrenchment benefits should an employer pay?",
    "Can an employer reject a flexible work arrangement request?",
    "What counts as wrongful dismissal in Singapore?"
]

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print('='*60)
    ask_hr_copilot(q)



Q: What is the notice period required for termination of employment?
ANSWER:
# Notice Period for Termination of Employment

According to the **Employment Act 1968**, the notice period required for termination of employment depends on the employee's length of service:

| Length of Service | Notice Period |
|---|---|
| Less than 26 weeks | 1 day |
| 26 weeks to less than 2 years | 1 week |
| 2 years to less than 5 years | 2 weeks |
| 5 years and above | 4 weeks |

**Source:** Employment Act 1968 - Section on "Notice Period for Termination of Employment" (as referenced in the context under mandatory retrenchment notification requirements)

**Note:** This applies as a minimum requirement under the Employment Act. The notice period applies when termination is initiated by either party (employer or employee).

SOURCES: ['workright-guide-employment-laws.pdf', 'tripartite-advisory-on-managing-excess-manpower-and-responsible-retrenchment.pdf', 'Retirement and Re-employment Act 1993.pdf', 'trip